<a href="https://colab.research.google.com/github/BilalKhaliqWillis/BILAL-Assignment2/blob/main/BILAL_Project_10_Movie_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommendation System

# Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

# Step 2: Load Dataset

In [4]:
movies = pd.read_csv('/content/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/tmdb_5000_credits.csv')

movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


# Step 3: Merge Dataset

In [26]:
credits.rename(columns={'movie_id':'id'}, inplace=True)
df = movies.merge(credits, on='id')

df.rename(columns={'title_x': 'title'}, inplace=True)

# Step 4: Data Cleaning

In [6]:
df['overview'] = df['overview'].fillna('')
df['popularity'] = df['popularity'].fillna(0)
df['vote_count'] = df['vote_count'].fillna(0)
df['vote_average'] = df['vote_average'].fillna(0)

df.isnull().sum()

,0
budget,0
genres,0
homepage,3091
id,0
keywords,0
original_language,0
original_title,0
overview,0
popularity,0
production_companies,0


# Step 5: Content-Based Filtering (TF-IDF)

In [7]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['overview'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Step 6: Recommendation Function (Content-Based)

In [28]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def recommend_content(title, num=5):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:num+1]
    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices]

recommend_content('Avatar')

,title
3604,Apollo 18
2130,The American
634,The Matrix
1341,The Inhabited Island
529,Tears of the Sun


# Step 7: Collaborative Filtering using SVD

In [29]:
features = df[['popularity', 'vote_count', 'vote_average', 'budget', 'revenue', 'runtime']]
features = features.fillna(0)

svd = TruncatedSVD(n_components=5)
latent_matrix = svd.fit_transform(features)

In [30]:
from sklearn.preprocessing import StandardScaler

features = df[['popularity', 'vote_count', 'vote_average', 'budget', 'revenue', 'runtime']]
features = features.fillna(0)

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

svd = TruncatedSVD(n_components=5)
latent_matrix = svd.fit_transform(features_scaled)

from sklearn.preprocessing import StandardScaler

features = df[['popularity', 'vote_count', 'vote_average', 'budget', 'revenue', 'runtime']]
features = features.fillna(0)

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

svd = TruncatedSVD(n_components=5)
latent_matrix = svd.fit_transform(features_scaled)

svd_sim = cosine_similarity(latent_matrix)

# Step 8: Recommendation Function (SVD)

In [31]:
def recommend_svd(title, num=5):
    idx = indices[title]
    sim_scores = list(enumerate(svd_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:num+1]
    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices]

recommend_svd('Avatar')

,title
233,Star Wars: Episode I - The Phantom Menace
31,Iron Man 3
16,The Avengers
216,Life of Pi
29,Skyfall


# Step 9: Autoencoder Model

In [32]:
input_dim = features.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(16, activation='relu')(input_layer)
encoded = Dense(8, activation='relu')(encoded)

decoded = Dense(16, activation='relu')(encoded)
decoded = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.fit(features, features, epochs=20, batch_size=32, verbose=1)

Epoch 1/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 7849409752596480.0000
Epoch 2/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 2218206726979584.0000
Epoch 3/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 253970392023040.0000
Epoch 4/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 11680352305152.0000
Epoch 5/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1775286353920.0000
Epoch 6/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1368510824448.0000
Epoch 7/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1139968311296.0000
Epoch 8/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1265265278976.0000
Epoch 9/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 999229554688.0000
Epoch 10/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 761900105728.0000
Epoch 11/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 661524054016.0000
Epoch 12/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 693358428160.0000
Epoch 13/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 

# Step 10: Autoencoder Recommendations

In [33]:
encoder = Model(input_layer, encoded)
encoded_movies = encoder.predict(features)

auto_sim = cosine_similarity(encoded_movies)

def recommend_autoencoder(title, num=5):
    idx = indices[title]
    sim_scores = list(enumerate(auto_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:num+1]
    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices]

recommend_autoencoder('Avatar')

151/151 ━━━━━━━━━━━━━━━━━━━━ 0s 837us/step


,title
330,The Lord of the Rings: The Two Towers
1359,Batman
4175,Beasts of the Southern Wild
0,Avatar
886,Mamma Mia!


# Step 11: Comparison

In [34]:
print('Content-Based Recommendations:')
print(recommend_content('Avatar'))

print('\nSVD Recommendations:')
print(recommend_svd('Avatar'))

print('\nAutoencoder Recommendations:')
print(recommend_autoencoder('Avatar'))

Content-Based Recommendations:
3604               Apollo 18
2130            The American
634               The Matrix
1341    The Inhabited Island
529         Tears of the Sun
Name: title, dtype: object

SVD Recommendations:
233    Star Wars: Episode I - The Phantom Menace
31                                    Iron Man 3
16                                  The Avengers
216                                   Life of Pi
29                                       Skyfall
Name: title, dtype: object

Autoencoder Recommendations:
330     The Lord of the Rings: The Two Towers
1359                                   Batman
4175              Beasts of the Southern Wild
0                                      Avatar
886                                Mamma Mia!
Name: title, dtype: object
